# Graph based analysis of interactions in Litigi community


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
from functions.utilities import *
from functions.analysis import *


## Load and preprocess data

In [ ]:
filename='processed_data\itigi_comments_copnsolidated.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

# extract the author of the parent comment (if applicable)
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)

litigi.head()

## Extract subsample

In [ ]:
litigi_subsample=litigi[:50000][['author','body_preprocessed','id', 'is_submitter', 'link_id', 'parent_id','created_dt','parent_author']]
litigi_subsample

## Comment chains

In [ ]:
C_Graph= create_nx_graph(litigi)
print(C_Graph)

In [ ]:
S = C_Graph.subgraph(max(nx.connected_components(C_Graph.to_undirected()), key=len)).copy()

In [ ]:
conversation=litigi[litigi['id'].isin(list(S.nodes))]
conversation[['id','parent_id']]

In [ ]:
conversation

## Extract user interactions

In [ ]:
user_interactions=litigi_subsample.groupby(['author','parent_author']).agg(**{'count':('id','count')}).reset_index()
user_interactions

In [ ]:
user_interactions=litigi_subsample.groupby(['author','parent_author']).agg(**{'count':('id','count')}).reset_index()
user_interactions

In [ ]:
G=nx.DiGraph()
G.add_edges_from([a for a in zip(user_interactions['author'],user_interactions['parent_author'],[{'weight': c} for c in user_interactions['count'] ])])
print(G)

In [ ]:
G_sym=symmetrize_graph(G)
log_w={k:np.log(nx.get_edge_attributes(G_sym,'weight')[k]) for k in nx.get_edge_attributes(G_sym,'weight')}


In [ ]:
pagerank = nx.pagerank(G.reverse())
print("Influencer Scores:", pagerank)
(dict(sorted(pagerank.items(), key=lambda item: item[1],reverse=True)))

In [ ]:
print(G)
print(G_sym)

In [ ]:
df=pd.DataFrame(nx.node_link_data(G_sym)['links'])
df

In [ ]:
G=extract_interaction_graph(litigi)
print(G)
pagerank = nx.pagerank(G)
print("Influencer Scores:", pagerank)
(dict(sorted(pagerank.items(), key=lambda item: item[1],reverse=True)))

In [ ]:
G_sym=symmetrize_graph(G)


In [ ]:
edge_weights = nx.get_edge_attributes(G_sym, 'weight')
log_w = {k: np.log(v) for k, v in edge_weights.items()}


## Export and visualize network

In [ ]:
from pyvis.network import Network


options = {
  "physics": {
    "enabled": False,
    "barnesHut": {
      "centralGravity": 1.6,
      "springLength": 140,
      "springConstant": 0.1
    },
    "minVelocity": 0.75,
    "timestep": 0.11
  }
}
nx.set_edge_attributes(G_sym,log_w,'weight')
# Convert NetworkX graph to Pyvis
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", select_menu=True)
net.show_buttons(filter_=['physics'])
net.from_nx(G_sym)
net.toggle_physics(False)
net.save_graph("output/litigi.html")

## Sources

https://towardsdatascience.com/large-graph-visualization-tools-and-approaches-2b8758a1cd59?gi=130d1c56065b

For a chord diagram look into: https://plotly.com/python/v3/chord-diagram/